# Implementations of 16 Qubit Random Number Generator
**Computer Laboratory II (417526) – Quantum AI – Practical 1**

Install once (Linux terminal):
```
pip install qiskit qiskit-aer matplotlib pylatexenc
```
Runs on the local Aer simulator – no IBM account or API token needed (the lab manual's `execute` / `IBMQ` / `qiskit.aqua` / `qiskit.ignis` code was removed from current Qiskit, so this notebook uses the current API).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__)

def run(qc, shots=1024, backend=sim, **kw):
    # Transpile for the backend, run, and return the counts dictionary.
    t = transpile(qc, backend, **kw)
    return backend.run(t, shots=shots).result().get_counts()

## Practical 1 – 16-Qubit Random Number Generator
**Idea:** A Hadamard gate puts a qubit in an equal superposition of |0⟩ and |1⟩. Measuring collapses it to 0 or 1 with probability 1/2 each. Doing this on 16 qubits gives 16 independent random bits, i.e. a number between 0 and 65535.

**Steps:** (1) create the quantum and classical registers → (2) build the circuit → (3) apply H to all qubits → (4) measure all qubits.

In [ ]:
q = QuantumRegister(16, 'q')          # Step 1: 16 qubits
c = ClassicalRegister(16, 'c')        #         16 classical bits
circuit = QuantumCircuit(q, c)        # Step 2: circuit
circuit.h(q)                          # Step 3: Hadamard on every qubit
circuit.measure(q, c)                 # Step 4: measure all qubits

display(circuit.draw(output='mpl'))

In [ ]:
counts = run(circuit, shots=1)        # one shot = one random 16-bit number
bits = list(counts.keys())[0]
print("RESULT (binary) :", bits)
print("RESULT (decimal):", int(bits, 2))

# Generate 5 more random numbers by taking several shots and reading memory
job = sim.run(transpile(circuit, sim), shots=5, memory=True)
for b in job.result().get_memory():
    print(b, "->", int(b, 2))